In [62]:
import os
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

In [63]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [64]:
dfs = []
folder = path + r"4. Working Data Files\Traffic Files\Capston_truck_country\truckstop_2024\State_export\\"

for filename in os.listdir(folder):
    if filename.lower().endswith(".csv"):
        file_path = os.path.join(folder, filename)
        # print("Reading:", file_path)
        df = pd.read_csv(file_path)
        df = df.groupby(["state_id", "f_system", "routenumber"]).agg({"aadt_combination": "sum"}).reset_index().copy()
        # break
        dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

In [65]:
combined_df.drop_duplicates(inplace=True, ignore_index=True)

In [66]:
state_df = pd.read_csv(path + "5. Source & Refrence Files\State_mapping.csv")

In [67]:
state_df[~state_df["state_code"].isin(combined_df["state_id"].unique())]

,State,state_code
1,AK,2
11,HI,15
39,PR,72
48,VI,78


In [68]:
combined_df

,state_id,f_system,routenumber,aadt_combination
0,1,1.0,10.0,6716165.0
1,1,1.0,20.0,14579350.0
2,1,1.0,22.0,7501743.0
3,1,1.0,59.0,23366146.0
4,1,1.0,65.0,34212874.0
...,...,...,...,...
343398,8,7.0,74.0,0.0
343399,8,7.0,84.0,0.0
343400,8,7.0,85.0,0.0
343401,8,7.0,109.0,0.0


In [69]:
aadt_vol = combined_df.groupby(["f_system", "routenumber"]).agg({"aadt_combination": "sum"}).reset_index().sort_values(
    ["aadt_combination"], ascending=False)

In [70]:
aadt_vol["aadt_pct"] = aadt_vol["aadt_combination"] / aadt_vol["aadt_combination"].sum()
aadt_vol["aadt_cum_pct"] = aadt_vol["aadt_pct"].cumsum()
aadt_vol = aadt_vol[aadt_vol["aadt_cum_pct"] <= .8].copy()
state_aadt_merge = aadt_vol.copy()

In [71]:
state_aadt_merge["key"] = state_aadt_merge["f_system"].astype("str") + state_aadt_merge["routenumber"].astype("str")

In [72]:
state_aadt_merge

,f_system,routenumber,aadt_combination,aadt_pct,aadt_cum_pct,key
79,1.0,80.0,370734358.0,0.060154,0.060154,1.080.0
9,1.0,10.0,289133321.0,0.046914,0.107067,1.010.0
39,1.0,40.0,266397750.0,0.043225,0.150292,1.040.0
74,1.0,75.0,218793478.0,0.035501,0.185793,1.075.0
69,1.0,70.0,218499323.0,0.035453,0.221245,1.070.0
...,...,...,...,...,...,...
2289,3.0,218.0,7035914.0,0.001142,0.795273,3.0218.0
354,1.0,710.0,6930537.0,0.001125,0.796397,1.0710.0
2154,3.0,83.0,6911813.0,0.001121,0.797519,3.083.0
2116,3.0,45.0,6853364.0,0.001112,0.798631,3.045.0


In [73]:
combined_df["key"] = combined_df["f_system"].astype("str") + combined_df["routenumber"].astype("str")
combined_df = combined_df[combined_df["key"].isin(state_aadt_merge["key"].unique())].copy()

In [74]:
combined_df_m = pd.merge(combined_df, state_df, left_on="state_id", right_on="state_code",
                         how="left")

In [75]:
combined_df_m[["routenumber", "f_system", "State"]].to_excel(
    path + r"3. Project Deliverables\3. TruckPath Comm\High Volume routes full country.xlsx", index=False)
